# Proposed validation — review before it runs

**What this measures:** The target metric is the ratio of trainable parameters between the published LoRA r=32 corpus row and a TernaryAdapt run, both produced by the repo's own MetaMathQA harness (llama-3.2-3B, rank32 protocol) — it directly quantifies the "far fewer trainable parameters than standard LoRA" half of the maintainer's claim (derived expectation ~32.8x: 9,174,720 LoRA vs 279,552 Ternary params across 28 q/v layer pairs). Because the claim names MetaMathQA and this repo publishes comparable numbers from that exact harness, the validation runs the real protocol on a dedicated GPU rather than the PR's synthetic-teacher CPU surrogate, which is not comparable to the published corpus.

**Target metric:** `param_ratio_lora_over_ternary`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at ``, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

## What will run

See `.remyx/validation.yaml` below — this proposal reuses an existing benchmark rather than adding a script.

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
# TernaryAdapt param-efficiency + fit-retention on the repo's OWN MetaMathQA method-comparison harness.
# The claim names MetaMathQA and the corpus already publishes lora--llama-3.2-3B-rank32.json from this
# harness, so suite kind (a) applies: run the REAL protocol (llama-3.2-3B SFT on MetaMathQA + eval) with
# ternary_adapt injected through the real get_peft_model entry point, and compare to the published LoRA row.
suite:
  harness:
    runner: method_comparison/MetaMathQA/run.py
    experiments: method_comparison/MetaMathQA
    results_glob: "method_comparison/MetaMathQA/results/*.json"
    method: lora

baseline:
  source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
  values:
    # Derived, not read: LoRA r=32 on llama-3.2-3B q/v targets = 28 layers x [32*(3072+3072) + 32*(3072+1024)]
    # = 28 x 327,680 = 9,174,720 trainable params. The corpus file is the source of record and overrides this.
    trainable_params: 9174720

metrics:
  # TARGET: trainable_params(lora--llama-3.2-3B-rank32) / trainable_params(ternary_adapt run).
  # Derivation: LoRA r=32 q/v = 9,174,720; TernaryAdapt default near-square Kronecker blocks
  # = 28 x [(64*64 + 48*48) + (32*64 + 32*48)] = 28 x (6,400 + 3,584) = 279,552; expected ratio ~32.8x.
  # Threshold 8.0 sits ~4x under the derivation and 8x above parity, so it fails any mask that is merely LoRA-sized.
  - name: param_ratio_lora_over_ternary
    direction: max
    threshold: 8.0
    role: target
  # GUARDRAIL: "without losing fit on MetaMathQA" — the ternary run's eval_loss may exceed the published LoRA
  # row's eval_loss (read from the corpus file at scoring time) by at most 3%. The pre-change baseline is the
  # LoRA corpus row itself (increase = 0.0), so it clears the bound by construction; 3% is a real fit bound, not vacuous.
  - name: eval_loss_increase_vs_lora_pct
    direction: min
    threshold: 3.0
    role: guardrail

policy:
  guardrail_veto: true

held_constant:
  - "base model: meta-llama/Llama-3.2-3B, same pinned revision the rank32 corpus runs used (gated, HF_TOKEN available)"
  - "training data and steps: the MetaMathQA configuration of the rank32 corpus runs, identical for both methods"
  - "target modules q_proj/v_proj for LoRA and TernaryAdapt alike (method_comparison protocol)"
  - "budget axis: LoRA r=32 vs TernaryAdapt default block_shape, both from the same 'rank32' corpus configuration"

avoid:
  - "unpinned model or dataset revisions; pin the llama-3.2-3B revision the corpus runs used"
  - "wall-clock or throughput gating on shared CPU (trainable-param counts and eval_loss are the instruments)"
  - "synthetic-teacher CPU surrogates reported as if they were the MetaMathQA protocol"

compute:
  # The claim is about fit on MetaMathQA with llama-3.2-3B: no CPU-valid instrument observes that; GPU it is.
  # timeout_s derived from one arm = LoRA-r32-comparable SFT of a 3B model on the corpus MetaMathQA subset
  # plus eval, the regime the published corpus rows were produced in: ~2-4 h on one dedicated GPU.
  tier: gpu
  timeout_s: 14400

provenance:
  param_ratio_lora_over_ternary: "user_guidance ('far fewer trainable parameters than standard LoRA')"
  eval_loss_increase_vs_lora_pct: "user_guidance ('without losing fit on MetaMathQA')"
  suite: "repo_runner:method_comparison/MetaMathQA/run.py"
  baseline: "published corpus method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
  held_constant: "protocol_doc:method_comparison/README.md"
  compute: "inferred (corpus regime: 3B SFT on MetaMathQA, one dedicated GPU per arm)"
  trainable_params: "inferred (derived from llama-3.2-3B q/v shapes at r=32; corpus file overrides)"
```